### Step 1: The Hybrid Feature Extraction Pipeline

**To build a robust, zero-day anomaly detection system, we cannot rely on a single mathematical perspective. We are building a "Hybrid Feature Matrix" that fuses human-interpretable Traditional Computer Vision with the complex pattern recognition of Deep Learning.**

Here is the exact breakdown of the five feature streams, what they do, how they work, and why we need them.

1. **Macro-Shape: Contour Fourier Descriptors (CFD)**
    * **What it is:** A mathematical representation of the object's outer boundary or silhouette.

    * **How we do it:** We convert the image to grayscale, extract the outermost continuous boundary (the contour), and map those X/Y coordinates as complex numbers. We then run a 1D Fast Fourier Transform (FFT) on that boundary to extract the frequency components, dropping the first component to make it position-invariant.

    * **Why we need it:** It allows the system to instantly recognize the global structural difference between a bipedal Human and a winged Bat, regardless of where they are standing on the screen or if they are rotated.

2. **Internal Geometry: Hu Moments**
    * **What it is:** A measure of the statistical center of mass and pixel weight distribution.

    * **How we do it:** We calculate image moments (weighted averages of pixel intensities) and derive 7 specific values. We then apply a logarithmic transform to compress the extreme mathematical variance.

    * **Why we need it:** Hu Moments are strictly invariant to scale, rotation, and reflection. If an Orc sprite is flipped horizontally or scaled down by 50%, its Hu Moments remain nearly identical. It helps us understand if the mass of the sprite is top-heavy, balanced, or dispersed.

3. **Color Palette: HSV Histogram**
    * **What it is:** A mapping of the material composition, skin tone, and armor paint of the sprite.

    * **How we do it:** Instead of standard RGB, we convert the image to HSV (Hue, Saturation, Value). We then mask out the empty background and build a 3D histogram strictly from the pixels inside the character's body.

    * **Why we need it:** RGB is terrible for machine learning because shadows change the RGB values completely. HSV separates the actual color (Hue) from the lighting (Value). This allows the math to know that a dark green Lizard in the shadows and a bright green Lizard in the light are the exact same species.

4. **Micro-Texture: Local Binary Patterns (LBP)**
    * **What it is:** A scanner for surface roughness, scales, armor joints, and shading edges.
    
    * **How we do it:** The algorithm looks at every single pixel and compares it to its 8 immediate neighbors in a 3x3 grid. If a neighbor is brighter, it scores a 1; if darker, a 0. This creates a binary number (texture fingerprint) for every pixel, which we then compile into a histogram.

    * **Why we need it:** Shape and color alone cannot tell the difference between a grey stone golem and a grey wizard in a cloth robe. LBP captures the high-frequency differences between smooth cloth, rigid metal, and rough scales.

5. **Deep Spatial Representation: Convolutional Autoencoder (CAE)**
    * **What it is:** A neural network that learns complex, non-linear structural relationships that traditional math misses.

    * **How we do it:** We pass the raw RGB image through multiple layers of convolutional filters, actively compressing the 80x80 image down into a dense 128-dimensional "bottleneck" vector. The network is trained to reconstruct the image perfectly from just those 128 numbers.

    * **Why we need it:** Hand-crafted features (CFD, LBP) are highly specific but rigidly defined. The CAE learns holistic context—like the fact that a "sword" usually appears near a "hand." It acts as the ultimate tie-breaker when the traditional CV features get confused.

#### **Dimensionality Reduction: Taming the Math**
Once we extract these 5 feature sets, we are left with thousands of numbers per image. If we feed raw, massive vectors into a clustering algorithm like GMM, it will suffer from the "Curse of Dimensionality" (where the math becomes so vast that everything looks equally far apart).

We solve this using a 2 methods
1. **PCA (Principal Component Analysis)**
    * **How it works:** PCA looks for the axes in the data that contain the highest amount of variance. It essentially squashes the data flat while trying to keep the most important differences intact. We set it to keep 95% of the mathematical variance.

    * **Why we use it:** It is fast, mathematically predictable, and preserves the "global" structure of the dataset. It is the perfect, stable baseline for traditional clustering.

2. **UMAP (Uniform Manifold Approximation and Projection)**
    * **How it works:** UMAP is a non-linear algorithm. Instead of drawing straight lines through the data, it builds a complex web connecting neighboring data points, and then tries to recreate that exact same web in a smaller 15-dimensional space.

    * **Why we use it:** UMAP is incredibly aggressive at pulling similar things into incredibly tight, dense "islands" while pushing dissimilar things far away. Because GMM and HDBSCAN are density-based clustering algorithms, feeding them UMAP's dense islands often results in vastly superior, cleaner clusters than PCA.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from skimage.feature import local_binary_pattern
from concurrent.futures import ThreadPoolExecutor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import cv2
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from skimage.feature import local_binary_pattern
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import concurrent.futures
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap



In [ ]:
# Traditional Feature Extractor

# 1. Applying Contour Fourier Descriptors to get Macro-Shape
def cfd(img):
    """
    Isolates the shape and converts the boundary into a frequency signal.
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(img_gray, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour = max(contours, key=cv2.contourArea) if contours else None
    if contour is None:
        return np.zeros(32).tolist()
    # 1D Fast Fourier Transform on the boundary coordinates
    comp_contour = contour[:, 0, 0] + 1j * contour[:, 0, 1]
    fourier_ = np.fft.fft(comp_contour)
    # Extracting magnitude (skipping the 0th DC component)
    mag = np.abs(fourier_)[1:33]
    if len(mag) < 32:
        mag = np.pad(mag, (0, 32 - len(mag)))
    return mag.tolist()

# 2. Applying Hu Moments to get Internal Geometry
def hum(img):
    """
    Calculates the statistical center of mass and pixel weight distribution.
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(img_gray, 1, 255, cv2.THRESH_BINARY) # Check the difference in results
    # Calculating Hu Moments
    moments = cv2.moments(mask)
    hu_moments = cv2.HuMoments(moments).flatten()
    # Logarithmic transform to compress mathematical variance
    hu_log = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)
    return hu_log.tolist()

# 3. Applying HSV Histogram to get Color Palette
def hsv(img):
    """
    Maps the material composition and paint of the image.
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(img_gray, 1, 255, cv2.THRESH_BINARY)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # Calculate 3D histogram strictly within the masked body
    hist = cv2.calcHist([img_hsv], [0, 1, 2], mask, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten().tolist()

# 4. Applying Local Binary Patterns to get Micro-Texture
def lbp(img):
    """
    Scans 3x3 pixel grids to capture surface roughness and shading edges.
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(img_gray, 1, 255, cv2.THRESH_BINARY)
    radius = 1
    n_points = 8 * radius
    # Computing LBP and isolate the object's body
    lbp = local_binary_pattern(img_gray, n_points, radius, method='uniform')
    lbp_masked = lbp[mask > 0]
    # Building translation-invariant histogram
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp_masked, bins=n_bins, range=(0, n_bins))
    hist = hist.astype('float')
    hist /= (hist.sum() + 1e-6)
    return hist.tolist()

In [ ]:
# Neural Network Feature Extractor

# CAE Architecture 
class CAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(CAE, self).__init__()      
        
        # Encoding - Compressing 80x80x3 -> 5x5x128
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), 
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), 
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Flatten(),
            nn.Linear(128 * 5 * 5, latent_dim)
        )

        # Decoding - Expanding  5x5x128 -> 80x80x3
        self.decoder_linear = nn.Sequential(
            nn.Linear(latent_dim, 128 * 5 * 5),
            nn.ReLU(True)
        )
        
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid() # Squashes pixels perfectly back to [0, 1] for RGB
        )

    def get_features(self, x):
        with torch.no_grad():
            return self.encoder(x)

    def forward(self, x):
        latent = self.encoder(x) # Compressing
        x_unflatten = self.decoder_linear(latent).view(-1, 128, 5, 5) # Reshaping for convolutions
        reconstructed = self.decoder_conv(x_unflatten) # Reconstructing
        return reconstructed

# Training
def cae(
    model,
    train_loader,
    epochs: int   = 50,
    lr: float = 1e-4,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    model.to(device)
    criterion     = nn.MSELoss()
    optimizer     = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler     = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode="min", factor=0.5, patience=5
                    )
    history = {"train_loss": []}
    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        running_loss = 0.0
        batch_count = 0
        for batch in train_loader:
            imgs = batch[0].to(device) if isinstance(batch, (list, tuple)) else batch.to(device)
            optimizer.zero_grad()
            reconstructed = model(imgs)
            loss = criterion(reconstructed, imgs)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            batch_count += 1

        epoch_loss = running_loss / len(train_loader.dataset)
        history["train_loss"].append(epoch_loss)
        scheduler.step(epoch_loss)
        print(f"Epoch [{epoch}/{epochs}] | Train Reconstruction Loss: {epoch_loss:.6f}")
    print("CAE feature extraction.")
    return history


In [ ]:
# Helper Functions

# PCA for Dimensionality Reduction
def PCA_(X, variance=0.95, name="Feature"):
    X = np.nan_to_num(X) # Catch any division-by-zero errors from Hu Moments
    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=variance, random_state=47)
    reduced = pca.fit_transform(X_scaled)
    print(f" - {name} reduced from {X.shape[1]}D to {reduced.shape[1]}D (Capturing {variance*100}% variance)")
    return reduced

# UMAP for Dimensionality Reduction
def UMAP_(X, target_dim=15, name="Feature"):
    X = np.nan_to_num(X)
    X_scaled = StandardScaler().fit_transform(X)
    
    # Safety Check: If the feature is already smaller than the target dimension (e.g., Hu Moments is 7D)
    # We bypass UMAP entirely and just return the scaled vector to prevent a crash.
    if X.shape[1] <= target_dim:
        print(f" - [UMAP BYPASS] {name} is already {X.shape[1]}D (<= target {target_dim}D). Bypassing UMAP.")
        return X_scaled
        
    reducer = umap.UMAP(n_components=target_dim, random_state=47)
    reduced = reducer.fit_transform(X_scaled)
    print(f" - [UMAP] {name} reduced from {X.shape[1]}D to {reduced.shape[1]}D")
    return reduced

In [ ]:
# Feature Pipeline

# Preparing data for CAE
def img_prep_cae(df, base_dir, img_size=80):
    print("Preparing Image Tensors for CAE")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    path_map = {}
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            path_map[file] = os.path.join(root, file)
    images = []
    for _, row in df.iterrows():
        path = path_map.get(row['image_name'])
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        img = np.transpose(img, (2, 0, 1)).astype(np.float32) / 255.0
        images.append(img)
    X_tensor = torch.tensor(np.array(images)).to(device)
    dataset = TensorDataset(X_tensor, X_tensor) 
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
    extract_loader = DataLoader(dataset, batch_size=256, shuffle=False)
    
    return dataloader, extract_loader, path_map

# Running on GPU
def extract_feature_cae(model, train_loader, extract_loader):
    print("Starting Convolutional Autoencoder")
    cae(model, train_loader, epochs=30, lr=1e-3)
    
    print("\n[GPU] Extracting 128D Latent Fingerprints...")
    model.eval()
    all_latents = []
    
    for batch in extract_loader:
        batch_data = batch[0]
        latent_tensors = model.get_features(batch_data)
        latent_flat = latent_tensors.view(latent_tensors.size(0), -1)
        all_latents.append(latent_flat.cpu().numpy())
        
    fused_deep_features = np.vstack(all_latents)
    print(f"CAE Extraction Complete! Shape: {fused_deep_features.shape}")
    return fused_deep_features

def process_single_image(path):
    img = cv2.imread(path)
    return {
        'cfd': cfd(img),
        'hum': hum(img),
        'hsv': hsv(img),
        'lbp': lbp(img)
    }

# Extracting features from Traditional CV methods
def extract_feature_cv(df, path_map):
    print(f"Starting Multithreaded CV Extraction for {len(df)} images...")
    paths = [path_map.get(name) for name in df['image_name']]
    results = []
    # Use ProcessPoolExecutor to truly bypass the Python GIL for heavy CPU math
    with concurrent.futures.ProcessPoolExecutor() as executor:
        # Map the function across all images, showing a progress bar
        for res in tqdm(executor.map(process_single_image, paths), total=len(paths), desc="CPU Extractor"):
            results.append(res)
    print("Traditional Extraction Complete!")
    # Unpack the dictionaries into clean numpy arrays
    f_cfd = np.array([r['cfd'] for r in results])
    f_hum = np.array([r['hum'] for r in results])
    f_hsv = np.array([r['hsv'] for r in results])
    f_lbp = np.array([r['lbp'] for r in results])
    return f_cfd, f_hum, f_hsv, f_lbp

# =======PIPELINE======
def feature_pipeline(base_dir, pca=True):

    # Build df by scanning base_dir for all valid image files
    records = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                stem      = file.rsplit('.', 1)[0]
                parts     = stem.rsplit('_', 1)
                category    = parts[0]
                subcategory = f"{parts[0]}_{parts[1][0]}"
                records.append({
                    'image_name':  file,
                    'category':    category,
                    'subcategory': subcategory
                })

    if not records:
        raise ValueError(f"No valid images found under '{base_dir}'.")

    df = pd.DataFrame(records)
    print(f"Discovered {len(df)} images in '{base_dir}'")

    # Preparing Data
    train_loader, extract_loader, path_map = img_prep_cae(df, base_dir)
    cae_model = CAE(latent_dim=128)

    # Ignite Simultaneous Execution
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as orchestrator:
        future_gpu = orchestrator.submit(extract_feature_cae, cae_model, train_loader, extract_loader)
        future_cpu = orchestrator.submit(extract_feature_cv, df, path_map)
        cae_features = future_gpu.result()
        f_cfd, f_hum, f_hsv, f_lbp = future_cpu.result()

    # Dimensionality Reduction Toggle
    mode_name = "PCA" if pca else "UMAP"
    print(f"\nAll Engines Finished. Initiating {mode_name} Compression...\n")

    if pca:
        comp_cae = PCA_(cae_features, name="CAE")
        comp_cfd = PCA_(f_cfd,        name="Macro-Shape")
        comp_hum = PCA_(f_hum,        name="Internal-Geometry")
        comp_hsv = PCA_(f_hsv,        name="Color Palette")
        comp_lbp = PCA_(f_lbp,        name="Texture")
    else:
        comp_cae = UMAP_(cae_features, target_dim=15, name="CAE")
        comp_cfd = UMAP_(f_cfd,        target_dim=15, name="Macro-Shape")
        comp_hum = UMAP_(f_hum,        target_dim=15, name="Internal-Geometry")
        comp_hsv = UMAP_(f_hsv,        target_dim=15, name="Color Palette")
        comp_lbp = UMAP_(f_lbp,        target_dim=15, name="Texture")

    # ==========================================
    # THE FUSION STEP
    # ==========================================
    # Fuse the 5 separate matrices horizontally into one master matrix
    x_final = np.hstack((comp_cae, comp_cfd, comp_hum, comp_hsv, comp_lbp))

    # Attach vectors as per-row objects (each cell holds a 1D numpy array)
    df = df.copy()
    
    # Add the single, ML-ready fused vector
    df['fused_features'] = list(x_final)
    
    # Keep the individual components just in case
    df['vec_cae'] = list(comp_cae)
    df['vec_cfd'] = list(comp_cfd)
    df['vec_hum'] = list(comp_hum)
    df['vec_hsv'] = list(comp_hsv)
    df['vec_lbp'] = list(comp_lbp)

    print(f"\nPipeline Complete. Master Feature Matrix successfully fused!")
    print(f"Final Fused Input Shape: {x_final.shape}")
    print(f"Returning DataFrame with shape {df.shape}")
    
    return df

In [ ]:
# EVALUTION FUNCTION
def full_evaluate(df, cluster_col):
    print(f" BENCHMARK: MICRO-CLUSTER PURITY REPORT ({cluster_col})")
    
    # Calculating Purity for every cluster
    clusters = sorted(df[cluster_col].unique())
    total_pure_clusters = 0
    
    for cid in clusters:
        # Isolating the images assigned to this specific cluster
        subset = df[df[cluster_col] == cid]
        total_in_cluster = len(subset)
        
        # Finding the dominant actual sprite sheet
        top_sheet = subset['sub_category'].mode()[0]
        top_sheet_count = len(subset[subset['sub_category'] == top_sheet])
        
        # Calculating how pure the cluster is
        purity = (top_sheet_count / total_in_cluster) * 100
        if purity == 100.0:
            total_pure_clusters += 1
        print(f"Cluster [{cid:>3}] | Total Images: {total_in_cluster:>4} | Purity: {purity:>6.2f}% | Dominant Sheet: '{top_sheet}' ({top_sheet_count}/{total_in_cluster})")
    print(f"\nPERFECT CLUSTERS: {total_pure_clusters} out of {len(clusters)} achieved 100% Micro-Purity")
    
    # HEATMAP
    print(f"\nMAPPING CLUSTERS TO CATEGORY BASED ON DOMINANT SPIECES")
    # normalize='index' ensures every row adds up to exactly 100%
    cluster_vs_macro = pd.crosstab(df[cluster_col], df['category'], normalize='index') * 100
    
    # Dynamically scale the height of the image based on the number of clusters
    fig_height = max(8, len(cluster_vs_macro) * 0.35)
    
    plt.figure(figsize=(10, fig_height))
    sns.heatmap(cluster_vs_macro, annot=True, fmt='.1f', cmap='mako', 
                cbar_kws={'label': '% of Cluster belonging to Category'})
    plt.title(f'Macro Purity Map: Raw Clusters vs Actual 5 Folders\n(Evaluated on: {cluster_col})')
    plt.ylabel('Raw Predicted GMM Cluster ID')
    plt.xlabel('Actual Truth Folder (5 Species)')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return total_pure_clusters/len(clusters)

def evaluate(df, cluster_col):
    """Silently calculates purity."""
    clusters = [c for c in df[cluster_col].unique() if c != -1]
    total_pure_clusters = 0
    
    for cid in clusters:
        subset = df[df[cluster_col] == cid]
        if len(subset) == 0: continue
        
        # FIX: Changed 'sub_category' to 'subcategory' to match the pipeline output
        top_sheet_count = len(subset[subset['subcategory'] == subset['subcategory'].mode()[0]])
        
        if (top_sheet_count / len(subset)) * 100 == 100.0:
            total_pure_clusters += 1
            
    ratio = (total_pure_clusters / len(clusters)) if len(clusters) > 0 else 0.0
    return total_pure_clusters, len(clusters), ratio

In [ ]:
if __name__ == "__main__":
    BASE_DIR = "./dungeon_images_colour80"
    
    df_pca = feature_pipeline(BASE_DIR, pca=True)
    df_umap = feature_pipeline(BASE_DIR, pca=False)

---
## Methods Applied to Classfy

In [ ]:
# Convert the fused column back to a Scikit-Learn friendly 2D matrix
X_pca = np.stack(df_pca['fused_features'].values)
X_umap = np.stack(df_umap['fused_features'].values)


comparison_df = pd.DataFrame(columns=['Method', 'Reducer', 'k', 'Pure', 'Total', 'Ratio'])

def log_results(comparison_df, method, reducer, k, total_pure, total_clusters, ratio):
    row = {
        'Method':  method,
        'Reducer': reducer,
        'k':       k,
        'Pure':    total_pure,
        'Total':   total_clusters,
        'Ratio':   ratio
    }
    return pd.concat([comparison_df, pd.DataFrame([row])], ignore_index=True)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

# 1. The Algorithm
def run_kmeans(X, k=51):
    if k > 0:
        return KMeans(n_clusters=k, random_state=47).fit_predict(X)
    else:
        print("   [Auto Search] Scanning k=30 to k=70 using Silhouette Score...")
        best_k, best_score, best_labels = 2, -1, None
        for i in range(30, 72, 2): 
            labels = KMeans(n_clusters=i, random_state=47).fit_predict(X)
            score = silhouette_score(X, labels)
            if score > best_score:
                best_k, best_score, best_labels = i, score, labels
        print(f"   [Auto Search] Math settled on optimal k = {best_k}")
        return best_labels

# 2. Initialize the Global Leaderboard (Run this only once at the top of your experiments)
if 'comparison_df' not in locals():
    comparison_df = pd.DataFrame(columns=['Method', 'Reducer', 'k', 'Pure', 'Total', 'Ratio'])

print("\n" + "="*50)
print(" EXPERIMENT 1: K-MEANS (Centroid-Based)")
print("="*50)

# --- 1. PCA | Forced k=51 ---
print("-> Running PCA | k=51...")
df_pca['pred_km_51'] = run_kmeans(X_pca, k=51)
p, t, r = evaluate(df_pca, 'pred_km_51')
comparison_df = log_results(comparison_df, 'K-Means', 'PCA', 51, p, t, r)

# --- 2. UMAP | Forced k=51 ---
print("\n-> Running UMAP | k=51...")
df_umap['pred_km_51'] = run_kmeans(X_umap, k=51)
p, t, r = evaluate(df_umap, 'pred_km_51')
comparison_df = log_results(comparison_df, 'K-Means', 'UMAP', 51, p, t, r)

# --- 3. PCA | Auto Search (k=0) ---
print("\n-> Running PCA | Auto Search...")
df_pca['pred_km_auto'] = run_kmeans(X_pca, k=0)
actual_k_pca = df_pca['pred_km_auto'].nunique() 
p, t, r = evaluate(df_pca, 'pred_km_auto')
comparison_df = log_results(comparison_df, 'K-Means', 'PCA', f"Auto ({actual_k_pca})", p, t, r)

# --- 4. UMAP | Auto Search (k=0) ---
print("\n-> Running UMAP | Auto Search...")
df_umap['pred_km_auto'] = run_kmeans(X_umap, k=0)
actual_k_umap = df_umap['pred_km_auto'].nunique()
p, t, r = evaluate(df_umap, 'pred_km_auto')
comparison_df = log_results(comparison_df, 'K-Means', 'UMAP', f"Auto ({actual_k_umap})", p, t, r)

comparison_df.head()

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

# 1. The GMM Algorithm
def run_gmm(X, k=51):
    if k > 0:
        return GaussianMixture(n_components=k, covariance_type='full', random_state=47).fit_predict(X)
    else:
        print("   [Auto Search] Scanning k=30 to k=70 using BIC (Lower is better)...")
        best_k, best_bic, best_labels = 2, np.inf, None
        
        # Test every k from 30 to 70 in steps of 2
        for i in range(30, 72, 2):
            gmm = GaussianMixture(n_components=i, covariance_type='full', random_state=47)
            labels = gmm.fit_predict(X)
            current_bic = gmm.bic(X)
            
            if current_bic < best_bic:
                best_k, best_bic, best_labels = i, current_bic, labels
                
        print(f"   [Auto Search] Math settled on optimal k = {best_k}")
        return best_labels

print("\n" + "="*50)
print(" EXPERIMENT 2: GMM (Distribution-Based)")
print("="*50)

# --- 1. PCA | Forced k=51 ---
print("-> Running PCA | k=51...")
df_pca['pred_gmm_51'] = run_gmm(X_pca, k=51)
p, t, r = evaluate(df_pca, 'pred_gmm_51')
comparison_df = log_results(comparison_df, 'GMM', 'PCA', 51, p, t, r)

# --- 2. UMAP | Forced k=51 ---
print("\n-> Running UMAP | k=51...")
df_umap['pred_gmm_51'] = run_gmm(X_umap, k=51)
p, t, r = evaluate(df_umap, 'pred_gmm_51')
comparison_df = log_results(comparison_df, 'GMM', 'UMAP', 51, p, t, r)

# --- 3. PCA | Auto Search (k=0) ---
print("\n-> Running PCA | Auto Search...")
df_pca['pred_gmm_auto'] = run_gmm(X_pca, k=0)
actual_k_pca = df_pca['pred_gmm_auto'].nunique() 
p, t, r = evaluate(df_pca, 'pred_gmm_auto')
comparison_df = log_results(comparison_df, 'GMM', 'PCA', f"Auto ({actual_k_pca})", p, t, r)

# --- 4. UMAP | Auto Search (k=0) ---
print("\n-> Running UMAP | Auto Search...")
df_umap['pred_gmm_auto'] = run_gmm(X_umap, k=0)
actual_k_umap = df_umap['pred_gmm_auto'].nunique()
p, t, r = evaluate(df_umap, 'pred_gmm_auto')
comparison_df = log_results(comparison_df, 'GMM', 'UMAP', f"Auto ({actual_k_umap})", p, t, r)

# ==========================================
# DISPLAY UPDATED LEADERBOARD
# ==========================================
print("\n" + "="*50)
print(" UPDATED RESULTS LEADERBOARD")
print("="*50)
# Using .to_string() here to bypass the tabulate dependency error!
print(comparison_df.to_string(index=False))

In [ ]:
import hdbscan
from sklearn.cluster import AgglomerativeClustering

# ==========================================
# 1. THE ALGORITHMS
# ==========================================
def run_hdbscan(X, k=0):
    # k is ignored. We set min_cluster_size to a reasonable baseline for 8000 images.
    print("   [Auto Search] HDBSCAN hunting for dense islands...")
    return hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5).fit_predict(X)

def run_agglomerative(X, k=51):
    if k > 0:
        return AgglomerativeClustering(n_clusters=k).fit_predict(X)
    else:
        print("   [Auto Search] Agglomerative building tree with distance threshold...")
        # distance_threshold forces n_clusters=None
        return AgglomerativeClustering(n_clusters=None, distance_threshold=15.0).fit_predict(X)

print("\n" + "="*50)
print(" EXPERIMENT 3: HDBSCAN & AGGLOMERATIVE")
print("="*50)

# --- 1. HDBSCAN (UMAP Strongly Preferred) ---
# HDBSCAN relies heavily on UMAP's topological density map
print("\n-> Running HDBSCAN | UMAP (Auto Only)...")
df_umap['pred_hdbscan'] = run_hdbscan(X_umap)
actual_k_hdb = len([c for c in df_umap['pred_hdbscan'].unique() if c != -1])
p, t, r = evaluate(df_umap, 'pred_hdbscan')
comparison_df = log_results(comparison_df, 'HDBSCAN', 'UMAP', f"Auto ({actual_k_hdb})", p, t, r)

# --- 2. AGGLOMERATIVE | Forced k=51 (PCA) ---
print("\n-> Running Agglomerative | PCA | k=51...")
df_pca['pred_agg_51'] = run_agglomerative(X_pca, k=51)
p, t, r = evaluate(df_pca, 'pred_agg_51')
comparison_df = log_results(comparison_df, 'Agglomerative', 'PCA', 51, p, t, r)

# --- 3. AGGLOMERATIVE | Forced k=51 (UMAP) ---
print("\n-> Running Agglomerative | UMAP | k=51...")
df_umap['pred_agg_51'] = run_agglomerative(X_umap, k=51)
p, t, r = evaluate(df_umap, 'pred_agg_51')
comparison_df = log_results(comparison_df, 'Agglomerative', 'UMAP', 51, p, t, r)

# --- 4. AGGLOMERATIVE | Auto Search (UMAP) ---
print("\n-> Running Agglomerative | UMAP | Auto Search...")
df_umap['pred_agg_auto'] = run_agglomerative(X_umap, k=0)
actual_k_agg = df_umap['pred_agg_auto'].nunique()
p, t, r = evaluate(df_umap, 'pred_agg_auto')
comparison_df = log_results(comparison_df, 'Agglomerative', 'UMAP', f"Auto ({actual_k_agg})", p, t, r)

# ==========================================
# DISPLAY UPDATED LEADERBOARD
# ==========================================
print("\n" + "="*50)
print(" UPDATED RESULTS LEADERBOARD")
print("="*50)
# Remember to use .to_string() so we don't trigger the tabulate error!
print(comparison_df.sort_values(by='Ratio', ascending=False).to_string(index=False))

In [ ]:
from sklearn.cluster import SpectralClustering, Birch
from sklearn.metrics import silhouette_score
try:
    from finch import FINCH
    finch_available = True
except ImportError:
    print("[!] finch-clust not installed. FINCH will be skipped.")
    finch_available = False

# ==========================================
# 1. THE ALGORITHMS
# ==========================================
def run_finch(X):
    print("   [Auto Search] FINCH finding parameter-free neighbor links...")
    c, num_clust, req_c = FINCH(X)
    print(f"   [Auto Search] FINCH naturally settled on k={num_clust[0]}")
    return c[:, 0] # Return the finest partition

def run_spectral(X, k=51):
    if k > 0:
        return SpectralClustering(n_clusters=k, random_state=47, n_jobs=-1, affinity='nearest_neighbors').fit_predict(X)
    else:
        print("   [Auto Search] Spectral scanning k=30 to k=70 using Silhouette...")
        best_k, best_score, best_labels = 2, -1, None
        for i in range(30, 72, 2):
            labels = SpectralClustering(n_clusters=i, random_state=47, n_jobs=-1, affinity='nearest_neighbors').fit_predict(X)
            score = silhouette_score(X, labels)
            if score > best_score:
                best_k, best_score, best_labels = i, score, labels
        print(f"   [Auto Search] Spectral settled on optimal k = {best_k}")
        return best_labels

def run_birch(X, k=51):
    if k > 0:
        return Birch(n_clusters=k).fit_predict(X)
    else:
        print("   [Auto Search] BIRCH building tree with threshold 0.5...")
        return Birch(n_clusters=None, threshold=0.5).fit_predict(X)

print("\n" + "="*50)
print(" EXPERIMENT 4: FINCH, SPECTRAL, & BIRCH")
print("="*50)

# --- 1. FINCH (Auto Only) ---
if finch_available:
    print("\n-> Running FINCH | PCA...")
    df_pca['pred_finch'] = run_finch(X_pca)
    actual_k_finch_pca = df_pca['pred_finch'].nunique()
    p, t, r = evaluate(df_pca, 'pred_finch')
    comparison_df = log_results(comparison_df, 'FINCH', 'PCA', f"Auto ({actual_k_finch_pca})", p, t, r)

    print("-> Running FINCH | UMAP...")
    df_umap['pred_finch'] = run_finch(X_umap)
    actual_k_finch_umap = df_umap['pred_finch'].nunique()
    p, t, r = evaluate(df_umap, 'pred_finch')
    comparison_df = log_results(comparison_df, 'FINCH', 'UMAP', f"Auto ({actual_k_finch_umap})", p, t, r)

# --- 2. SPECTRAL CLUSTERING ---
print("\n-> Running Spectral | UMAP | k=51...")
df_umap['pred_spec_51'] = run_spectral(X_umap, k=51)
p, t, r = evaluate(df_umap, 'pred_spec_51')
comparison_df = log_results(comparison_df, 'Spectral', 'UMAP', 51, p, t, r)

print("\n-> Running Spectral | UMAP | Auto Search...")
df_umap['pred_spec_auto'] = run_spectral(X_umap, k=0)
actual_k_spec = df_umap['pred_spec_auto'].nunique()
p, t, r = evaluate(df_umap, 'pred_spec_auto')
comparison_df = log_results(comparison_df, 'Spectral', 'UMAP', f"Auto ({actual_k_spec})", p, t, r)

# --- 3. BIRCH ---
print("\n-> Running BIRCH | PCA | k=51...")
df_pca['pred_birch_51'] = run_birch(X_pca, k=51)
p, t, r = evaluate(df_pca, 'pred_birch_51')
comparison_df = log_results(comparison_df, 'BIRCH', 'PCA', 51, p, t, r)

print("\n-> Running BIRCH | PCA | Auto Search...")
df_pca['pred_birch_auto'] = run_birch(X_pca, k=0)
actual_k_birch = df_pca['pred_birch_auto'].nunique()
p, t, r = evaluate(df_pca, 'pred_birch_auto')
comparison_df = log_results(comparison_df, 'BIRCH', 'PCA', f"Auto ({actual_k_birch})", p, t, r)

# ==========================================
# DISPLAY UPDATED LEADERBOARD
# ==========================================
print("\n" + "="*50)
print(" UPDATED RESULTS LEADERBOARD")
print("="*50)
print(comparison_df.sort_values(by='Ratio', ascending=False).to_string(index=False))

In [ ]:
from sklearn.cluster import DBSCAN, OPTICS, MeanShift, estimate_bandwidth, AffinityPropagation
import numpy as np

# ==========================================
# 1. THE ALGORITHMS
# ==========================================
def run_dbscan(X):
    print("   [Auto Search] DBSCAN scanning for dense regions...")
    # eps is the maximum distance between two samples for them to be considered in the same neighborhood
    return DBSCAN(eps=0.5, min_samples=5, n_jobs=-1).fit_predict(X)

def run_optics(X):
    print("   [Auto Search] OPTICS scanning for variable density regions...")
    return OPTICS(min_samples=10, n_jobs=-1).fit_predict(X)

def run_meanshift(X):
    print("   [Auto Search] Mean Shift calculating density centers (This takes a moment)...")
    # Subsampling to 1000 to prevent RAM overflow during bandwidth estimation
    bandwidth = estimate_bandwidth(X, quantile=0.2, n_samples=1000)
    return MeanShift(bandwidth=bandwidth, bin_seeding=True, n_jobs=-1).fit_predict(X)

def run_affinity(X):
    print("   [Auto Search] Affinity Propagation passing messages (WARNING: Highly intensive)...")
    return AffinityPropagation(random_state=47).fit_predict(X)

print("\n" + "="*50)
print(" EXPERIMENT 5: THE FINAL FOUR")
print("="*50)

# --- 1. DBSCAN (UMAP Preferred for Density) ---
print("\n-> Running DBSCAN | UMAP...")
df_umap['pred_dbscan'] = run_dbscan(X_umap)
# DBSCAN uses -1 for noise, we must exclude it from the actual k count
k_dbscan = len([c for c in df_umap['pred_dbscan'].unique() if c != -1])
p, t, r = evaluate(df_umap, 'pred_dbscan')
comparison_df = log_results(comparison_df, 'DBSCAN', 'UMAP', f"Auto ({k_dbscan})", p, t, r)

# --- 2. OPTICS (UMAP Preferred for Density) ---
print("\n-> Running OPTICS | UMAP...")
df_umap['pred_optics'] = run_optics(X_umap)
k_optics = len([c for c in df_umap['pred_optics'].unique() if c != -1])
p, t, r = evaluate(df_umap, 'pred_optics')
comparison_df = log_results(comparison_df, 'OPTICS', 'UMAP', f"Auto ({k_optics})", p, t, r)

# --- 3. MEAN SHIFT (PCA Preferred for Centroids) ---
print("\n-> Running Mean Shift | PCA...")
df_pca['pred_meanshift'] = run_meanshift(X_pca)
k_ms = len([c for c in df_pca['pred_meanshift'].unique() if c != -1])
p, t, r = evaluate(df_pca, 'pred_meanshift')
comparison_df = log_results(comparison_df, 'Mean Shift', 'PCA', f"Auto ({k_ms})", p, t, r)

# --- 4. AFFINITY PROPAGATION (OPTIONAL) ---
# Uncomment the block below ONLY if you have 16GB+ RAM and a few minutes to spare!
"""
print("\n-> Running Affinity Propagation | PCA...")
df_pca['pred_affinity'] = run_affinity(X_pca)
k_aff = len([c for c in df_pca['pred_affinity'].unique() if c != -1])
p, t, r = evaluate(df_pca, 'pred_affinity')
comparison_df = log_results(comparison_df, 'Affinity', 'PCA', f"Auto ({k_aff})", p, t, r)
"""

# ==========================================
# THE GRAND FINALE LEADERBOARD
# ==========================================
print("\n" + "="*50)
print(" 🏆 THE ULTIMATE RESULTS LEADERBOARD 🏆")
print("="*50)
print(comparison_df.sort_values(by='Ratio', ascending=False).to_string(index=False))

In [ ]:
# ==========================================
# THE GRAND FINALE LEADERBOARD
# ==========================================
print("\n" + "="*50)
print(" 🏆 THE ULTIMATE RESULTS LEADERBOARD 🏆")
print("="*50)
print(comparison_df.sort_values(by='Ratio', ascending=False).to_string(index=False))